# Intrinsic Wavefront Analysis and Plots

**Author:** Aaron Roodman  
**Date Created:** 2026-02-23  
**Last Modified:** 2026-03-17  
**Status:** In Progress  
**Keywords:** AOS, Intrinsic Wavefront, Full Array Mode, Zernike, Analysis

## Description

This notebook analyzes the FAM Zernike table created by `intrinsics_mktable.ipynb`.

**Analysis includes:**
1. Load parquet file with Zernike measurements and visit_info companion table
2. Identify FAM blocks (contiguous triplets with visits separated by 3) and print summary
3. Filter out sparse days (< 5 FAM triplet seq_num)
4. Robust per-image linear fit (constant + x,y slopes) to data-model residuals using RLM
5. Plot fit parameters and residual histograms per day_obs
6. Single-image residual maps and histograms for Z4-Z11
7. Create hexbin visualizations: Data, Model, and Data-Model residuals

**Input:** Parquet files created by mktable notebook (in `output/`)  
**Output:** Trio comparison plots (PNG), fit parameter plots, single-image residual maps, FAM block summary

**Based on:** `intrinsics_mktable.ipynb`

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-02-23 | Aaron Roodman | Initial version |
| 2026-03-15 | Aaron Roodman | Added CCS/OCS coord_sys parameter; per-image linear fit; fit parameter plots; rotator angle filter |
| 2026-03-17 | Aaron Roodman | Switched to robust RLM; FAM block analysis; single-image residual maps; MPEG movie; multi-page PDFs |
| 2026-03-19 | Aaron Roodman | Generalized focal-plane Zernike fit method with Noll convention basis (normalized to fp_radius=1.75 deg); two fits per image: z1toz3 (Noll 1-3: piston+tilt+tip) and z1toz6 (Noll 1-6: adds defocus+astigmatism); fit output includes coefficients, bse errors, RLM scale per Zernike; merged output table (intrinsic_fits_ parquet) with visit_info; trio plots use median and go into single PDF; clean up individual JPEGs after successful movie creation |
| 2026-03-19 | Aaron Roodman | Read Noll indices from visit_info nollIndices column instead of guessing; all plots and fit column names use actual Noll indices from data; added z1toz6 fit parameter plots alongside z1toz3; plot Zernike sets derived from data (iZs_plot_12) instead of hardcoded [4..15] |
| 2026-03-19 | Aaron Roodman | Renamed fit prefixes z13→z1toz3, z16→z1toz6 to avoid confusion with Zernike 13/16; added error bars to fit coefficient plots; all coefficient units are μm (basis is normalized to fp_radius) |
| 2026-03-25 | Aaron Roodman | Widened rotator angle filter to ±90°; added az/el/rot to single-image residual titles; grouped telescope positions (±3° tolerance) with distinct markers in fit coefficient plots; tightened colorbar spacing in residual maps |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Load Data](#load)
4. [FAM Block Analysis](#blocks)
5. [Helper Functions](#functions)
6. [Analysis](#analysis)
7. [Focal-Plane Zernike Fits](#fit)
8. [Output Fit Table](#fittable)
9. [Fit Parameter Plots & Residual Histograms](#fitplots)
10. [Single-Image Residual Maps](#singleimage)
11. [Trio Comparison Plots](#viz)

<a id='params'></a>
## Parameters

In [ ]:
from pathlib import Path

# ============================================================
# Parameters — All configurable values collected here
# ============================================================

# Coordinate system: 'CCS' or 'OCS' (must match mktable choice)
coord_sys = 'OCS'

# Parquet file version strings (must match mktable output)
prefix = 'fam_danish'
wep_ver = 'wep_v16_8_0'
dviz_ver = 'dviz_v3_5_0'

# Parquet file dates
day_obs_min = 20260315
day_obs_max = 20260317

# Date range string for plot titles
date_range_str = f'{day_obs_min}-{day_obs_max}'

# Rotator angle filter (degrees) — only keep images within this range
rotator_angle_min = -90.0
rotator_angle_max = 90.0

# Minimum unique seq_num per day_obs to keep (filters sparse days)
min_seq_num_per_day = 5

# Tolerance for grouping telescope positions (az, el, rot) in degrees
position_group_tol = 3.0

# Bad fit threshold: flag visits with any |coefficient| > this value (μm)
bad_fit_threshold = 2.0

# Zernike indices: inferred from data after loading (see Load Data section)
# iZs and iZidx are set dynamically below

# Plotting parameters
plo_default = 4.0   # Low percentile for colorbar
phi_default = 96.0  # High percentile for colorbar
output_dir = 'output'

# Per-run output subdirectory (unique name from all parameters)
run_dir = f'{output_dir}/{prefix}_{coord_sys}_{wep_ver}_{dviz_ver}_{day_obs_min}_{day_obs_max}'

# Ensure output directories exist
Path(output_dir).mkdir(parents=True, exist_ok=True)
Path(run_dir).mkdir(parents=True, exist_ok=True)
print(f"Run output directory: {run_dir}")

<a id='setup'></a>
## Setup & Imports

In [2]:
# Standard imports
from matplotlib import pyplot as plt
from matplotlib import lines
from matplotlib.backends.backend_pdf import PdfPages
from mpl_toolkits import axes_grid1
from matplotlib.colors import LinearSegmentedColormap

import numpy as np
import pandas as pd
from scipy.stats import binned_statistic_2d, binned_statistic
from pathlib import Path
import statsmodels.api as sm
import subprocess

# Astropy
from astropy.table import Table, QTable

# Set plot defaults
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Pandas display options
pd.options.display.max_columns = None
pd.options.display.width = None

<a id='load'></a>
## Load Data

In [ ]:
# Load the main donut parquet file and visit_info companion table
parquet_file = f'{output_dir}/{prefix}_zernikes_{wep_ver}_{dviz_ver}_{day_obs_min}_{day_obs_max}.parquet'
visit_info_file = f'{output_dir}/{prefix}_zernikes_{wep_ver}_{dviz_ver}_{day_obs_min}_{day_obs_max}_visits.parquet'

if not Path(parquet_file).exists():
    raise FileNotFoundError(f"Parquet file not found: {parquet_file}\n"
                           f"Run intrinsics_mktable.ipynb first to create it.")

print(f"Loading data from: {parquet_file}")
aosTable = QTable.read(parquet_file)
print(f"Loaded {len(aosTable)} rows, {len(aosTable.columns)} columns")
print(f"Coordinate system: {coord_sys}")

# Load visit_info table
if Path(visit_info_file).exists():
    visit_info = QTable.read(visit_info_file)
    print(f"\nLoaded visit_info from: {visit_info_file}")
    print(f"  {len(visit_info)} visits, columns: {visit_info.colnames}")
    
    # Convert az, alt from radians to degrees
    visit_info['az'] = np.rad2deg(np.array(visit_info['az']))
    visit_info['alt'] = np.rad2deg(np.array(visit_info['alt']))
    print(f"  Converted az, alt from radians to degrees")
    print(f"  az range: [{visit_info['az'].min():.1f}, {visit_info['az'].max():.1f}] deg")
    print(f"  alt range: [{visit_info['alt'].min():.1f}, {visit_info['alt'].max():.1f}] deg")
    
    # Get rotator_angle from the main donut table (in degrees) instead of
    # visit_info's rotAngle/skyAngle which is a different angle
    if 'rotator_angle' in aosTable.columns:
        vi_dobs = np.array(visit_info['day_obs'])
        vi_snum = np.array(visit_info['seq_num'])
        main_dobs = np.array(aosTable['day_obs'])
        main_snum = np.array(aosTable['seq_num'])
        main_rot = np.array(aosTable['rotator_angle'])
        # Compute mean rotator_angle per visit from the main table
        rot_per_visit = []
        for i in range(len(visit_info)):
            mask = (main_dobs == vi_dobs[i]) & (main_snum == vi_snum[i])
            if np.sum(mask) > 0:
                rot_per_visit.append(float(np.mean(main_rot[mask])))
            else:
                rot_per_visit.append(np.nan)
        visit_info['rotAngle'] = rot_per_visit
        print(f"  rotAngle from main table (rotator_angle, degrees)")
        print(f"  rotAngle range: [{np.nanmin(rot_per_visit):.1f}, "
              f"{np.nanmax(rot_per_visit):.1f}] deg")
    else:
        print(f"  Warning: rotator_angle not in main table, "
              f"keeping visit_info rotAngle as-is")
else:
    visit_info = None
    print(f"\nWarning: visit_info file not found: {visit_info_file}")
    print("  FAM block analysis will be skipped.")

In [4]:
# Display table summary
print("\nTable columns:")
print(sorted(aosTable.columns))

# Count by day_obs
print("\nMeasurements per day_obs:")
day_counts = pd.DataFrame({'day_obs': aosTable['day_obs']}).value_counts().sort_index()
print(day_counts)


Table columns:
['centroid_x', 'centroid_x_extra', 'centroid_x_intra', 'centroid_y', 'centroid_y_extra', 'centroid_y_intra', 'coord_dec', 'coord_ra', 'day_obs', 'detector', 'extra_fpx', 'extra_fpy', 'intra_fpx', 'intra_fpy', 'matched_intra_extra', 'rotator_angle', 'rotator_flagged', 'seq_num', 'snr', 'snr_extra', 'snr_intra', 'thx_CCS', 'thx_CCS_extra', 'thx_CCS_intra', 'thx_OCS', 'thx_OCS_extra', 'thx_OCS_intra', 'thy_CCS', 'thy_CCS_extra', 'thy_CCS_intra', 'thy_OCS', 'thy_OCS_extra', 'thy_OCS_intra', 'used', 'zk_CCS', 'zk_OCS', 'zk_OCS_mean', 'zk_deviation_CCS', 'zk_deviation_OCS', 'zk_intrinsic', 'zk_intrinsic_CCS', 'zk_intrinsic_OCS', 'zk_residual']

Measurements per day_obs:
day_obs 
20260315    177835
20260316     87310
20260317    168632
Name: count, dtype: int64


In [5]:
# Display sample rows with formatted floats
# Select scalar columns only
scalar_cols = []
for col in aosTable.columns:
    if not hasattr(aosTable[col][0], '__len__') or isinstance(aosTable[col][0], str):
        scalar_cols.append(col)

df_display = aosTable[scalar_cols].to_pandas()
pd.options.display.float_format = '{:.2f}'.format

print("\nSample rows (first 5):")
print(df_display.head())

pd.reset_option('display.float_format')


Sample rows (first 5):
   used detector  coord_ra  coord_dec  centroid_x  centroid_y  thx_CCS  \
0  True  R01_S01      2.23      -0.47     2758.48     3343.97    -0.03   
1  True  R01_S01      2.23      -0.47     3733.60     3671.67    -0.03   
2  True  R01_S02      2.23      -0.46     1695.68     1936.54    -0.03   
3  True  R01_S02      2.23      -0.47     2574.27     3679.61    -0.03   
4  True  R01_S02      2.23      -0.47     1368.32     3285.68    -0.03   

   thy_CCS  thx_OCS  thy_OCS      snr  centroid_x_intra  centroid_x_extra  \
0    -0.01    -0.03    -0.01 31168.08           2758.65           2758.31   
1    -0.01    -0.03    -0.01 25022.96           3644.92           3822.28   
2    -0.01    -0.03    -0.01 53468.31           1697.26           1694.11   
3    -0.01    -0.03    -0.01 51147.34           2576.05           2572.48   
4    -0.01    -0.03    -0.01 50303.31           1369.64           1367.00   

   centroid_y_intra  centroid_y_extra  thx_CCS_intra  thx_CCS_extra 

<a id='blocks'></a>
## FAM Block Analysis

Identify contiguous blocks of Full Array Mode (FAM) triplets from the visit_info table.
FAM triplets (intra, in-focus, extra-focal) have visits separated by increments of 3.

In [6]:
# Identify FAM blocks from visit_info
# FAM triplets (intra, in-focus, extra) have visits separated by increments of 3.
# A FAM block is a contiguous group of visits where consecutive visits differ by exactly 3.

if visit_info is not None:
    visits_sorted = visit_info.copy()
    visits_sorted.sort('visit')
    visit_arr = np.array(visits_sorted['visit'])
    
    # Get mean Z4 (first element of zk_OCS) per visit from the main table
    day_obs_main = np.array(aosTable['day_obs'])
    seq_num_main = np.array(aosTable['seq_num'])
    zk_data_all = np.stack(aosTable[f'zk_{coord_sys}'])
    
    # Compute mean Z4 per visit
    visit_z4_mean = {}
    for row in visits_sorted:
        dobs = row['day_obs']
        snum = row['seq_num']
        mask = (day_obs_main == dobs) & (seq_num_main == snum)
        if np.sum(mask) > 0:
            visit_z4_mean[(dobs, snum)] = np.mean(zk_data_all[mask, 0])
        else:
            visit_z4_mean[(dobs, snum)] = np.nan
    
    # Find block boundaries: consecutive visits differ by 3
    diffs = np.diff(visit_arr)
    block_breaks = np.where(diffs != 3)[0] + 1
    block_starts = np.concatenate([[0], block_breaks])
    block_ends = np.concatenate([block_breaks, [len(visit_arr)]])
    
    print(f"Found {len(block_starts)} FAM blocks from {len(visit_arr)} visits\n")
    
    # Build summary table
    block_summary = []
    for b_idx, (bs, be) in enumerate(zip(block_starts, block_ends)):
        block_rows = visits_sorted[bs:be]
        n_visits = len(block_rows)
        dobs = int(block_rows['day_obs'][0])
        seq_min = int(np.min(block_rows['seq_num']))
        seq_max = int(np.max(block_rows['seq_num']))
        band = str(block_rows['band'][0])
        alt_mean = float(np.mean(block_rows['alt']))
        az_mean = float(np.mean(block_rows['az']))
        rot_mean = float(np.mean(block_rows['rotAngle']))
        
        # Z4 of first and last image in block
        first_key = (int(block_rows['day_obs'][0]), int(block_rows['seq_num'][0]))
        last_key = (int(block_rows['day_obs'][-1]), int(block_rows['seq_num'][-1]))
        z4_first = visit_z4_mean.get(first_key, np.nan)
        z4_last = visit_z4_mean.get(last_key, np.nan)
        
        block_summary.append({
            'block': b_idx,
            'day_obs': dobs,
            'seq_min': seq_min,
            'seq_max': seq_max,
            'n_visits': n_visits,
            'band': band,
            'alt': alt_mean,
            'az': az_mean,
            'rotAngle': rot_mean,
            'Z4_first': z4_first,
            'Z4_last': z4_last,
        })
    
    block_df = pd.DataFrame(block_summary)
    pd.options.display.float_format = '{:.2f}'.format
    print("FAM Block Summary:")
    print(block_df.to_string(index=False))
    pd.reset_option('display.float_format')
else:
    print("visit_info not available — skipping FAM block analysis")

Found 16 FAM blocks from 178 visits

FAM Block Summary:
 block  day_obs  seq_min  seq_max  n_visits band  alt   az  rotAngle  Z4_first  Z4_last
     0 20260315       73      106        12    r 1.23 1.57      2.94     -0.08    -0.38
     1 20260315      122      161        14    i 1.22 4.71      4.68     -0.45    -0.35
     2 20260315      184      217        12    i 1.22 4.71      1.24      0.01    -0.37
     3 20260315      230      263        12    i 1.22 4.71      0.20     -0.29    -0.62
     4 20260315      276      309        12    i 1.22 4.71      6.22     -0.05    -0.19
     5 20260315      322      355        12    i 1.22 4.71      0.46     -0.11    -0.16
     6 20260316      647      680        12    i 1.22 4.71      5.43      0.06    -0.34
     7 20260316      693      726        12    i 1.22 4.71      1.24      0.07    -0.38
     8 20260316      737      767        11    i 1.22 4.71      0.20     -0.01    -0.23
     9 20260317       64       97        12    i 1.22 6.27      

In [7]:
# Filter out day_obs with fewer than min_seq_num_per_day unique seq_num
# These are days with too few FAM triplets for reliable analysis
day_obs_all = np.array(aosTable['day_obs'])
seq_num_all = np.array(aosTable['seq_num'])

unique_days = sorted(set(day_obs_all.tolist()))
print(f"Before filtering: {len(unique_days)} days, {len(aosTable)} donuts")
print(f"Unique seq_num per day_obs (minimum required: {min_seq_num_per_day}):")

sparse_days = []
for d in unique_days:
    n_unique_seq = len(set(seq_num_all[day_obs_all == d].tolist()))
    n_donuts_day = np.sum(day_obs_all == d)
    status = "REMOVED" if n_unique_seq < min_seq_num_per_day else "kept"
    print(f"  {d}: {n_unique_seq} seq_num, {n_donuts_day} donuts  [{status}]")
    if n_unique_seq < min_seq_num_per_day:
        sparse_days.append(d)

if sparse_days:
    sparse_mask = ~np.isin(day_obs_all, sparse_days)
    n_before = len(aosTable)
    aosTable = aosTable[sparse_mask]
    n_after = len(aosTable)
    print(f"\nRemoved {len(sparse_days)} day_obs with < {min_seq_num_per_day} unique seq_num: {sparse_days}")
    print(f"  Donuts: {n_before} -> {n_after} ({n_before - n_after} removed)")
else:
    print(f"\nAll day_obs have >= {min_seq_num_per_day} unique seq_num")

# Summary of remaining data
remaining_days = sorted(set(np.array(aosTable['day_obs']).tolist()))
n_images = len(set(zip(np.array(aosTable['day_obs']).tolist(), np.array(aosTable['seq_num']).tolist())))
print(f"\nRemaining: {len(remaining_days)} days, {n_images} images, {len(aosTable)} donuts")

Before filtering: 3 days, 433777 donuts
Unique seq_num per day_obs (minimum required: 5):
  20260315: 74 seq_num, 177835 donuts  [kept]
  20260316: 35 seq_num, 87310 donuts  [kept]
  20260317: 69 seq_num, 168632 donuts  [kept]

All day_obs have >= 5 unique seq_num

Remaining: 3 days, 178 images, 433777 donuts


In [8]:
# Filter by rotator angle
if 'rotator_angle' in aosTable.columns:
    rot_angles = np.array(aosTable['rotator_angle'])
    rot_mask = (rot_angles >= rotator_angle_min) & (rot_angles <= rotator_angle_max)
    n_before = len(aosTable)
    aosTable = aosTable[rot_mask]
    n_after = len(aosTable)
    n_images_before = len(set(zip(np.array(aosTable['day_obs']), np.array(aosTable['seq_num']))))
    print(f"Rotator angle filter [{rotator_angle_min}, {rotator_angle_max}] deg:")
    print(f"  Donuts: {n_before} -> {n_after} ({n_before - n_after} removed)")
    print(f"  Remaining images: {n_images_before}")
else:
    print("Warning: 'rotator_angle' column not found in table.")
    print("  Rotator angle filter NOT applied.")
    print("  Re-run intrinsics_mktable.ipynb to include rotator angles.")

Rotator angle filter [-3.0, 3.0] deg:
  Donuts: 433777 -> 119711 (314066 removed)
  Remaining images: 49


<a id='functions'></a>
## Helper Functions

In [ ]:
def get_zernike(table, column_name, iZ):
    """Extract a single Zernike term from an array column.
    
    Parameters
    ----------
    table : QTable
        Table with Zernike array columns
    column_name : str
        Column name (e.g. 'zk_OCS', 'zk_intrinsic_OCS', 'zk_residual')
    iZ : int
        Zernike index (4-28, excluding 20, 21)
    
    Returns
    -------
    array : ndarray
        Zernike values
    """
    if iZ not in iZidx:
        raise ValueError(f"Zernike Z{iZ} not in table. Available: {iZs}")
    zk_array = np.stack(table[column_name])
    return zk_array[:, iZidx[iZ]]


def add_colorbar(im, aspect=20, pad_fraction=0.5, **kwargs):
    """Add a vertical color bar to an image plot."""
    divider = axes_grid1.make_axes_locatable(im.axes)
    width = axes_grid1.axes_size.AxesY(im.axes, aspect=1./aspect)
    pad = axes_grid1.axes_size.Fraction(pad_fraction, width)
    current_ax = plt.gca()
    cax = divider.append_axes("right", size=width, pad=pad)
    plt.sca(current_ax)
    return im.axes.figure.colorbar(im, cax=cax, **kwargs)


# ============================================================
# Focal-plane Zernike fitting
# ============================================================

def focal_plane_zernike_basis(thx_deg, thy_deg, max_noll, fp_radius=1.75):
    """Build focal-plane Noll Zernike basis matrix.
    
    Coordinates are normalized to the focal-plane radius so that the
    Zernike polynomials are evaluated on a unit disk.  All basis functions
    are dimensionless, so fit coefficients have the same units as the data (μm).
    
    Parameters
    ----------
    thx_deg, thy_deg : ndarray
        Field angles in degrees.
    max_noll : int
        Maximum Noll index (1-6 supported).
    fp_radius : float
        Focal plane radius in degrees for normalization (default 1.75).
    
    Returns
    -------
    A : ndarray, shape (n_points, max_noll)
        Design matrix with one column per focal Zernike term.
    labels : list of str
        Labels for each column (e.g. 'Z1_piston', 'Z2_tilt', ...).
    """
    x = thx_deg / fp_radius
    y = thy_deg / fp_radius
    r2 = x**2 + y**2
    
    cols = []
    labels = []
    
    if max_noll >= 1:
        cols.append(np.ones_like(x))
        labels.append('Z1_piston')
    if max_noll >= 2:
        cols.append(2.0 * x)
        labels.append('Z2_tilt')
    if max_noll >= 3:
        cols.append(2.0 * y)
        labels.append('Z3_tip')
    if max_noll >= 4:
        cols.append(np.sqrt(3) * (2.0 * r2 - 1.0))
        labels.append('Z4_defocus')
    if max_noll >= 5:
        cols.append(2.0 * np.sqrt(6) * x * y)
        labels.append('Z5_astig45')
    if max_noll >= 6:
        cols.append(np.sqrt(6) * (x**2 - y**2))
        labels.append('Z6_astig0')
    
    return np.column_stack(cols), labels


def fit_focal_zernikes(aosTable_matched, iZs, iZidx, coord_sys,
                        max_focal_noll=3, include_intrinsic=True,
                        fp_radius=1.75, prefix='z1toz3',
                        zk_intrinsic_col='zk_intrinsic_OCS'):
    """Fit focal-plane Noll Zernikes to per-image data residuals using robust regression.
    
    For each image and each Zernike term iZ, fits:
        residual = k1*Zfocal_1 + k2*Zfocal_2 + ... + kN*Zfocal_N
    where residual = zk_data - zk_intrinsic*1e6 (if include_intrinsic) or just zk_data.
    
    Parameters
    ----------
    aosTable_matched : QTable
        Matched donut table.
    iZs : list of int
        Zernike indices to fit.
    iZidx : dict
        Mapping from iZ to column index in zk arrays.
    coord_sys : str
        Coordinate system ('OCS' or 'CCS').
    max_focal_noll : int
        Maximum focal Noll index (default 3 for piston+tilt+tip).
    include_intrinsic : bool
        If True, subtract intrinsic model before fitting (default True).
    fp_radius : float
        Focal plane radius in degrees (default 1.75).
    prefix : str
        Column name prefix for output (e.g. 'z1toz3' or 'z1toz6').
    zk_intrinsic_col : str
        Column name for intrinsic Zernikes (e.g. 'zk_intrinsic_OCS').
    
    Returns
    -------
    fit_table : QTable
        One row per image with columns:
        - day_obs, seq_num, image_idx, n_donuts
        - {prefix}_z{iZ}_c{j} : fit coefficient j for Zernike iZ (μm)
        - {prefix}_z{iZ}_c{j}_err : standard error of coefficient (μm)
        - {prefix}_z{iZ}_scale : RLM scale estimate (robust sigma, μm)
    zk_fit_vals : ndarray, shape (n_donuts, n_zernikes)
        Per-donut fitted values.
    zk_rlm_weights : ndarray, shape (n_donuts, n_zernikes)
        Per-donut RLM weights.
    """
    thx_col = f'thx_{coord_sys}'
    thy_col = f'thy_{coord_sys}'
    
    day_obs_arr = np.array(aosTable_matched['day_obs'])
    seq_num_arr = np.array(aosTable_matched['seq_num'])
    thx_arr = np.rad2deg(np.array(aosTable_matched[thx_col]))
    thy_arr = np.rad2deg(np.array(aosTable_matched[thy_col]))
    zk_data = np.stack(aosTable_matched[f'zk_{coord_sys}'])
    zk_model = np.stack(aosTable_matched[zk_intrinsic_col])
    
    images = sorted(set(zip(day_obs_arr.tolist(), seq_num_arr.tolist())))
    n_donuts = len(aosTable_matched)
    n_zernikes = len(iZs)
    n_coeffs = max_focal_noll
    
    zk_fit_vals = np.zeros((n_donuts, n_zernikes))
    zk_rlm_weights = np.ones((n_donuts, n_zernikes))
    fit_params_list = []
    
    for img_idx, (dobs, snum) in enumerate(images):
        mask = (day_obs_arr == dobs) & (seq_num_arr == snum)
        n_pts = np.sum(mask)
        
        # Build design matrix
        A, col_labels = focal_plane_zernike_basis(
            thx_arr[mask], thy_arr[mask], max_focal_noll, fp_radius)
        
        img_params = {'day_obs': dobs, 'seq_num': snum,
                      'image_idx': img_idx, 'n_donuts': int(n_pts)}
        
        for j_idx, iZ in enumerate(iZs):
            # Compute residual
            if include_intrinsic:
                resid = zk_data[mask, j_idx] - zk_model[mask, j_idx] * 1e6
            else:
                resid = zk_data[mask, j_idx].copy()
            
            # Robust fit
            try:
                rlm_model = sm.RLM(resid, A, M=sm.robust.norms.HuberT())
                rlm_results = rlm_model.fit()
                coeffs = rlm_results.params
                bse = rlm_results.bse
                scale = float(rlm_results.scale)
                weights = rlm_results.weights
            except Exception:
                coeffs, _, _, _ = np.linalg.lstsq(A, resid, rcond=None)
                bse = np.full(n_coeffs, np.nan)
                scale = float(np.std(resid - A @ coeffs))
                weights = np.ones(n_pts)
            
            # Store coefficients, errors, scale
            for ci in range(n_coeffs):
                img_params[f'{prefix}_z{iZ}_c{ci+1}'] = float(coeffs[ci])
                img_params[f'{prefix}_z{iZ}_c{ci+1}_err'] = float(bse[ci])
            img_params[f'{prefix}_z{iZ}_scale'] = scale
            
            # Per-donut fitted values
            zk_fit_vals[mask, j_idx] = A @ coeffs
            zk_rlm_weights[mask, j_idx] = weights
        
        fit_params_list.append(img_params)
    
    fit_table = QTable(fit_params_list)
    print(f"Fit '{prefix}' (focal Noll 1-{max_focal_noll}): "
          f"{len(images)} images, {n_donuts} donuts, "
          f"include_intrinsic={include_intrinsic}")
    
    return fit_table, zk_fit_vals, zk_rlm_weights


# ============================================================
# Plotting methods
# ============================================================

def _fmt_angle(v):
    """Format angle for display: 2 sig figs, no scientific notation."""
    if v == 0:
        return '0'
    s = f'{v:.2g}'
    if 'e' in s:
        return f'{v:.0f}'
    return s


def _build_pointing_groups(fit_table_day, visit_info, bin_size=3.0, verbose=False):
    """Build pointing groups by binning (az, alt, rotAngle) to nearest bin_size degrees.
    
    Azimuth is wrapped to [-180, 180] before binning so that values near
    0/360 boundary (e.g. 358° and 2°) land in the same group.
    
    Parameters
    ----------
    fit_table_day : QTable
        Fit table with day_obs, seq_num columns.
    visit_info : QTable
        Visit metadata with az, alt, rotAngle columns.
    bin_size : float
        Bin width in degrees for rounding (default 3.0).
    verbose : bool
        If True, print a summary table of the groups.
    
    Returns
    -------
    group_indices : dict
        Maps group_key (az_bin, alt_bin, rot_bin) -> list of row indices
    group_labels : dict
        Maps group_key -> display label string
    """
    ft_dobs = np.array(fit_table_day['day_obs'])
    ft_snum = np.array(fit_table_day['seq_num'])
    
    # Build visit_info lookup
    vi_lookup = {}
    vi_dobs = np.array(visit_info['day_obs'])
    vi_snum = np.array(visit_info['seq_num'])
    vi_az = np.array(visit_info['az'])
    vi_alt = np.array(visit_info['alt'])
    vi_rot = np.array(visit_info['rotAngle'])
    for i in range(len(visit_info)):
        vi_lookup[(int(vi_dobs[i]), int(vi_snum[i]))] = (
            float(vi_az[i]), float(vi_alt[i]), float(vi_rot[i]))
    
    group_indices = {}
    # Track raw (az, alt, rot) per group for summary
    group_raw_values = {}
    for idx in range(len(fit_table_day)):
        key = (int(ft_dobs[idx]), int(ft_snum[idx]))
        if key in vi_lookup:
            az, alt, rot = vi_lookup[key]
            # Wrap azimuth to [-180, 180] so 358° and 2° both map near 0°
            az_wrapped = ((az + 180) % 360) - 180
            gkey = (round(az_wrapped / bin_size) * bin_size,
                    round(alt / bin_size) * bin_size,
                    round(rot / bin_size) * bin_size)
        else:
            az, alt, rot = np.nan, np.nan, np.nan
            gkey = (np.nan, np.nan, np.nan)
        group_indices.setdefault(gkey, []).append(idx)
        group_raw_values.setdefault(gkey, []).append((az, alt, rot))
    
    group_labels = {}
    for gkey in group_indices:
        if np.isnan(gkey[0]):
            group_labels[gkey] = 'unknown'
        else:
            group_labels[gkey] = (f'az={_fmt_angle(gkey[0])} '
                                  f'el={_fmt_angle(gkey[1])} '
                                  f'rot={_fmt_angle(gkey[2])}')
    
    if verbose:
        print(f"\nPointing groups (bin_size={bin_size}°):")
        print(f"{'Group':>6s}  {'Label':30s}  {'N_visits':>8s}  "
              f"{'Az range':>12s}  {'El range':>12s}  {'Rot range':>12s}")
        print('-' * 95)
        for gi, gkey in enumerate(sorted(group_indices.keys(),
                                         key=lambda g: group_indices[g][0])):
            n = len(group_indices[gkey])
            raw = np.array(group_raw_values[gkey])
            az_lo, az_hi = raw[:, 0].min(), raw[:, 0].max()
            el_lo, el_hi = raw[:, 1].min(), raw[:, 1].max()
            rot_lo, rot_hi = raw[:, 2].min(), raw[:, 2].max()
            print(f'{gi:6d}  {group_labels[gkey]:30s}  {n:8d}  '
                  f'{az_lo:5.1f}-{az_hi:5.1f}  '
                  f'{el_lo:5.1f}-{el_hi:5.1f}  '
                  f'{rot_lo:6.1f}-{rot_hi:5.1f}')
        print()
    
    return group_indices, group_labels


def plot_fit_params_and_residuals(fit_table_day, aosTable_matched, plot_mask_day,
                                  day_obs_list, fit_prefix='z1toz3',
                                  iZs_fit_plot=None, iZs_hist=None,
                                  visit_info=None,
                                  zk_intrinsic_col='zk_intrinsic_OCS',
                                  output_dir='.', show=True):
    """Plot fit coefficients vs image (with error bars and pointing-group markers)
    and residual histograms.
    
    Parameters
    ----------
    fit_table_day : QTable
        Fit parameters table filtered to selected days.
    aosTable_matched : QTable
        Full matched donut table (with zk_fit column).
    plot_mask_day : ndarray (bool)
        Mask into aosTable_matched for the selected days.
    day_obs_list : list of int
        Day_obs values being plotted (used in titles).
    fit_prefix : str
        Prefix for fit columns (e.g. 'z1toz3').
    iZs_fit_plot : list of int, optional
        Zernike indices for fit param plots.
    iZs_hist : list of int, optional
        Zernike indices for residual histograms.
    visit_info : QTable, optional
        If provided, use (az, alt, rotAngle) to assign marker symbols per pointing group.
    zk_intrinsic_col : str
        Column name for intrinsic Zernikes (e.g. 'zk_intrinsic_OCS').
    output_dir : str
        Directory for saved figures.
    show : bool
        If True, display plots inline. If False, save and close.
    """
    if iZs_fit_plot is None:
        iZs_fit_plot = [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
    if iZs_hist is None:
        iZs_hist = [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
    
    if len(day_obs_list) == 1:
        day_label = str(day_obs_list[0])
    elif len(day_obs_list) <= 4:
        day_label = ', '.join(str(d) for d in day_obs_list)
    else:
        day_label = f'{day_obs_list[0]}...{day_obs_list[-1]} ({len(day_obs_list)} days)'
    
    file_suffix = str(day_obs_list[0]) if len(day_obs_list) == 1 else 'all'
    
    # Determine how many coefficients this fit has
    first_iZ = iZs_fit_plot[0]
    sample_cols = [c for c in fit_table_day.colnames
                   if c.startswith(f'{fit_prefix}_z{first_iZ}_c') and not c.endswith('_err')]
    n_coeffs = len(sample_cols)
    
    param_labels = ['k1 (piston)', 'k2 (tilt)', 'k3 (tip)',
                    'k4 (defocus)', 'k5 (astig45)', 'k6 (astig0)'][:n_coeffs]
    param_units = ['μm'] * n_coeffs
    
    # Compute grid dimensions from input list length
    n_plots = len(iZs_fit_plot)
    ncols = 4
    nrows = (n_plots + ncols - 1) // ncols  # ceil division
    
    # Build pointing groups if visit_info available
    markers = ['o', 's', '^', 'D', 'v', 'P', '*', 'X', 'p', 'h']
    if visit_info is not None and len(fit_table_day) > 0:
        group_indices, group_labels = _build_pointing_groups(fit_table_day, visit_info, bin_size=position_group_tol)
        # Sort groups by first index for consistent ordering
        sorted_groups = sorted(group_indices.keys(), key=lambda g: group_indices[g][0])
    else:
        group_indices = None
    
    # Determine label suffix for higher-order vs standard Zernike sets
    is_hi = (iZs_fit_plot[0] > 15) if len(iZs_fit_plot) > 0 else False
    hi_suffix = '_hi' if is_hi else ''
    pdf_path = f'{output_dir}/fit_params_resid_{fit_prefix}{hi_suffix}_{file_suffix}.pdf'
    
    with PdfPages(pdf_path) as pdf:
        image_idx = np.arange(len(fit_table_day))
        
        for ci in range(n_coeffs):
            cname = f'c{ci+1}'
            kname = f'k{ci+1}'
            fig, axes = plt.subplots(nrows, ncols, figsize=(18, 10 * nrows / 3), sharex=True)
            if nrows == 1:
                axes = axes[np.newaxis, :]  # ensure 2D indexing
            fig.suptitle(f'{fit_prefix} Fit: {param_labels[ci]} vs Image (day_obs: {day_label})',
                         fontsize=14)
            
            for ax_idx, iZ in enumerate(iZs_fit_plot):
                ax = axes[ax_idx // ncols, ax_idx % ncols]
                col = f'{fit_prefix}_z{iZ}_{cname}'
                err_col = f'{fit_prefix}_z{iZ}_{cname}_err'
                if col not in fit_table_day.colnames:
                    continue
                vals = np.array(fit_table_day[col])
                errs = (np.array(fit_table_day[err_col])
                        if err_col in fit_table_day.colnames else None)
                
                if group_indices is not None and len(sorted_groups) > 1:
                    # Draw connecting line for all points first
                    ax.plot(image_idx, vals, '-', color='gray', linewidth=0.5,
                            alpha=0.5, zorder=1)
                    # Plot each group with its own marker
                    for gi, gkey in enumerate(sorted_groups):
                        idxs = np.array(group_indices[gkey])
                        m = markers[gi % len(markers)]
                        label = group_labels[gkey] if ax_idx == 0 else None
                        if errs is not None:
                            ax.errorbar(image_idx[idxs], vals[idxs], yerr=errs[idxs],
                                        fmt=m, markersize=4, linewidth=0,
                                        elinewidth=0.5, capsize=0, alpha=0.8,
                                        label=label, zorder=2)
                        else:
                            ax.plot(image_idx[idxs], vals[idxs], m, markersize=4,
                                    alpha=0.8, label=label, zorder=2)
                else:
                    # Single group or no visit_info — simple plot
                    if errs is not None:
                        ax.errorbar(image_idx, vals, yerr=errs,
                                    fmt='o-', markersize=3, linewidth=0.8,
                                    elinewidth=0.5, capsize=0, alpha=0.8)
                    else:
                        ax.plot(image_idx, vals, 'o-', markersize=3)
                
                ax.set_title(f'Z{iZ}')
                ax.set_ylabel(param_units[ci])
                ax.axhline(0, color='gray', linestyle='--', linewidth=0.5)
                if ax_idx // ncols == nrows - 1:
                    ax.set_xlabel('Image index')
            
            # Hide unused axes
            for idx in range(n_plots, nrows * ncols):
                axes[idx // ncols, idx % ncols].set_visible(False)
            
            # Add legend on first subplot if we have groups
            if group_indices is not None and len(sorted_groups) > 1:
                axes[0, 0].legend(fontsize=6, loc='best', ncol=1,
                                  handletextpad=0.3, borderpad=0.3)
            
            plt.tight_layout()
            pdf.savefig(fig, dpi=150, bbox_inches='tight')
            if show:
                plt.show()
            else:
                plt.close(fig)
        
        # Fit residual histograms
        zk_data_plot = np.stack(aosTable_matched[f'zk_{coord_sys}'])[plot_mask_day]
        zk_model_plot = np.stack(aosTable_matched[zk_intrinsic_col])[plot_mask_day]
        fit_col = f'zk_fit_{fit_prefix}'
        if fit_col in aosTable_matched.colnames:
            zk_fit_plot = np.stack(aosTable_matched[fit_col])[plot_mask_day]
        else:
            zk_fit_plot = np.stack(aosTable_matched['zk_fit'])[plot_mask_day]
        
        n_hist = len(iZs_hist)
        nrows_h = (n_hist + ncols - 1) // ncols
        fig, axes = plt.subplots(nrows_h, ncols, figsize=(18, 10 * nrows_h / 3))
        if nrows_h == 1:
            axes = axes[np.newaxis, :]
        fig.suptitle(f'{fit_prefix} Fit Residuals: data - model - fit (day_obs: {day_label})', fontsize=14)
        
        hist_range = (-1.0, 1.0)
        n_bins = 100
        
        for ax_idx, iZ in enumerate(iZs_hist):
            ax = axes[ax_idx // ncols, ax_idx % ncols]
            j = iZidx[iZ]
            resid = zk_data_plot[:, j] - zk_model_plot[:, j] * 1e6 - zk_fit_plot[:, j]
            n_total = len(resid)
            n_in = np.sum((resid >= hist_range[0]) & (resid <= hist_range[1]))
            n_out = n_total - n_in
            ax.hist(resid, bins=n_bins, range=hist_range, log=True,
                    edgecolor='black', linewidth=0.3, alpha=0.7)
            ax.set_title(f'Z{iZ}')
            ax.set_xlabel('μm')
            ax.set_ylabel('Count')
            ax.set_xlim(hist_range)
            std_val = np.std(resid)
            ax.text(0.97, 0.95, f'σ={std_val:.3f} μm\n{n_out}/{n_total} outside',
                    transform=ax.transAxes, ha='right', va='top', fontsize=8,
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        # Hide unused histogram axes
        for idx in range(n_hist, nrows_h * ncols):
            axes[idx // ncols, idx % ncols].set_visible(False)
        
        plt.tight_layout()
        pdf.savefig(fig, dpi=150, bbox_inches='tight')
        if show:
            plt.show()
        else:
            plt.close(fig)
    
    print(f"Saved: {pdf_path}")


def plot_single_image_residual_grid(aosTable_matched, day_obs, seq_num,
                                     band='', alt=None, az=None, rotAngle=None,
                                     fit_table=None, fit_prefix='z1toz3',
                                     iZs_plot=None,
                                     zk_intrinsic_col='zk_intrinsic_OCS',
                                     n_steps=16, statistic='median',
                                     plo=2.0, phi=98.0,
                                     k1_range=1.0, fpradius=1.8,
                                     clim_dict=None,
                                     output_dir='.', show=True):
    """Plot a 3x4 grid of binned residual maps for a single image.

    Uses subplots with axes_grid1 colorbars attached to each axis,
    ensuring colorbars stay tight against their image and never overlap
    neighboring y-axis labels.  Includes k1 (piston) bar indicator.
    """
    from matplotlib.patches import Rectangle

    if iZs_plot is None:
        iZs_plot = [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]

    dobs_arr = np.array(aosTable_matched['day_obs'])
    snum_arr = np.array(aosTable_matched['seq_num'])
    img_mask = (dobs_arr == day_obs) & (snum_arr == seq_num)
    n_donuts = np.sum(img_mask)
    if n_donuts == 0:
        return None

    # Look up k1 (piston) values from fit_table
    k1_vals = {}
    if fit_table is not None:
        ft_dobs = np.array(fit_table['day_obs'])
        ft_snum = np.array(fit_table['seq_num'])
        ft_mask = (ft_dobs == day_obs) & (ft_snum == seq_num)
        if np.sum(ft_mask) == 1:
            ft_row = fit_table[ft_mask][0]
            for iZ in iZs_plot:
                col = f'{fit_prefix}_z{iZ}_c1'
                if col in fit_table.colnames:
                    k1_vals[iZ] = float(ft_row[col])

    xval = np.rad2deg(np.array(aosTable_matched[f'thy_{coord_sys}_extra'])[img_mask])
    yval = np.rad2deg(np.array(aosTable_matched[f'thx_{coord_sys}_extra'])[img_mask])
    zk_data_img = np.stack(aosTable_matched[f'zk_{coord_sys}'])[img_mask]
    zk_model_img = np.stack(aosTable_matched[zk_intrinsic_col])[img_mask]
    fit_col = f'zk_fit_{fit_prefix}'
    if fit_col in aosTable_matched.colnames:
        zk_fit_img = np.stack(aosTable_matched[fit_col])[img_mask]
    else:
        zk_fit_img = np.stack(aosTable_matched['zk_fit'])[img_mask]

    bins_edge = np.linspace(-fpradius, fpradius, n_steps)
    n_rows, n_cols = 3, 4

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 12))

    # Build title with metadata
    band_str = f'  band={band}' if band else ''
    meta_str = ''
    if az is not None:
        meta_str = f'  az={az:.1f}  el={alt:.1f}  rot={rotAngle:.1f}'
    fig.suptitle(f'Single-Image Residuals: day_obs={day_obs}  seq_num={seq_num}'
                 f'{band_str}{meta_str}  ({n_donuts} donuts)',
                 fontsize=13, y=0.98)

    for ax_idx, iZ in enumerate(iZs_plot):
        row = ax_idx // n_cols
        col = ax_idx % n_cols
        ax = axes[row, col]
        j = iZidx[iZ]
        resid = zk_data_img[:, j] - zk_model_img[:, j] * 1e6 - zk_fit_img[:, j]

        stat_val, _, _, _ = binned_statistic_2d(
            xval, yval, resid, statistic=statistic, bins=[bins_edge, bins_edge])
        if clim_dict is not None and iZ in clim_dict:
            vmin, vmax = clim_dict[iZ]
        else:
            finite_vals = stat_val[np.isfinite(stat_val)]
            vmin, vmax = (np.percentile(finite_vals, [plo, phi]) if len(finite_vals) > 0
                          else (-1.0, 1.0))

        im = ax.imshow(stat_val.T, origin='lower',
                       extent=[-fpradius, fpradius, -fpradius, fpradius],
                       cmap='RdBu_r', interpolation='none', aspect='equal',
                       vmin=vmin, vmax=vmax)
        # Colorbar attached directly to axis via axes_grid1 — stays tight
        add_colorbar(im, aspect=25, pad_fraction=0.3)
        ax.set_title(f'Z{iZ}', fontsize=11, loc='center')
        ax.set_xlim(-fpradius, fpradius)
        ax.set_ylim(-fpradius, fpradius)
        if col == 0:
            ax.set_ylabel(f'thx_{coord_sys} [deg]')
        if row == n_rows - 1:
            ax.set_xlabel(f'thy_{coord_sys} [deg]')

        # k1 (piston) bar indicator
        if iZ in k1_vals:
            k1_val = np.clip(k1_vals[iZ], -k1_range, k1_range)
            bar_anchor, bar_y, bar_h = 0.25, 1.04, 0.03
            bar_dx = k1_val / k1_range * 0.25
            color = 'steelblue' if k1_val >= 0 else 'firebrick'
            rect_x = min(bar_anchor, bar_anchor + bar_dx)
            rect_w = abs(bar_dx)
            if rect_w > 0.001:
                rect = Rectangle((rect_x, bar_y - bar_h / 2), rect_w, bar_h,
                                 transform=ax.transAxes, clip_on=False,
                                 facecolor=color, edgecolor='none', alpha=0.7)
                ax.add_patch(rect)
            ax.plot([bar_anchor, bar_anchor], [bar_y - bar_h / 2, bar_y + bar_h / 2],
                    transform=ax.transAxes, color='black', linewidth=0.8, clip_on=False)
            ax.text(bar_anchor + bar_dx, bar_y + bar_h / 2 + 0.01,
                    f'{k1_vals[iZ]:.2f}', transform=ax.transAxes,
                    fontsize=7, ha='center', va='bottom', color=color)

    plt.tight_layout()
    output_file = f'{output_dir}/single_image_resid_{day_obs}_{seq_num}.jpg'
    fig.savefig(output_file, dpi=120, bbox_inches='tight')
    if show:
        plt.show()
    else:
        plt.close(fig)
    return output_file


def plot_zernike_trio(aosTable_matched, iZ, plo=4.0, phi=96.0,
                      statistic='median', fit_prefix='z1toz3',
                      zk_intrinsic_col='zk_intrinsic_OCS',
                      output_dir='.', date_range_str='', pdf=None):
    """Create trio of plots for a Zernike term: Data, Model, Data-Model.
    
    Parameters
    ----------
    statistic : str
        Statistic for binned_statistic_2d (default 'median').
    zk_intrinsic_col : str
        Column name for intrinsic Zernikes (e.g. 'zk_intrinsic_OCS').
    pdf : PdfPages or None
        If provided, save figure to this PDF instead of a standalone PNG.
    """
    nsteps = 18 * 4 + 1
    fpradius = 1.8
    xbins = np.linspace(-fpradius, fpradius, nsteps)
    ybins = np.linspace(-fpradius, fpradius, nsteps)
    
    xval = np.rad2deg(aosTable_matched[f'thy_{coord_sys}_extra'])
    yval = np.rad2deg(aosTable_matched[f'thx_{coord_sys}_extra'])
    
    zk_data_all = np.stack(aosTable_matched[f'zk_{coord_sys}'])
    fit_col = f'zk_fit_{fit_prefix}'
    if fit_col in aosTable_matched.colnames:
        zk_fit_all = np.stack(aosTable_matched[fit_col])
    else:
        zk_fit_all = np.stack(aosTable_matched['zk_fit'])
    zk_model_all = np.stack(aosTable_matched[zk_intrinsic_col])
    
    zval_data = zk_data_all[:, iZidx[iZ]] - zk_fit_all[:, iZidx[iZ]]
    zval_model = zk_model_all[:, iZidx[iZ]] * 1e6
    zval_residual = zval_data - zval_model
    
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    
    for ax, zval, cmap, title_str in [
        (axes[0], zval_data, 'viridis', f'Z{iZ} Data (linear fit subtracted)'),
        (axes[1], zval_model, 'viridis', f'Z{iZ} Model Intrinsic'),
        (axes[2], zval_residual, 'RdBu_r', f'Z{iZ} Data - Model'),
    ]:
        stat_val, _, _, _ = binned_statistic_2d(
            xval, yval, zval, statistic=statistic, bins=[xbins, ybins])
        vmin, vmax = np.nanpercentile(zval, [plo, phi])
        im = ax.imshow(stat_val.T, origin='lower',
                       extent=[xbins[0], xbins[-1], ybins[0], ybins[-1]],
                       cmap=cmap, interpolation='none', aspect='equal',
                       vmin=vmin, vmax=vmax)
        add_colorbar(im, label='μm')
        ax.set_xlabel(f'thy_{coord_sys} [deg]')
        ax.set_ylabel(f'thx_{coord_sys} [deg]')
        ax.set_title(title_str)
        ax.set_aspect('equal')
    
    title = f'Z{iZ} Comparison ({statistic})'
    if date_range_str:
        title += f' ({date_range_str})'
    fig.suptitle(title, fontsize=14, y=0.98)
    plt.tight_layout()
    
    if pdf is not None:
        pdf.savefig(fig, dpi=150, bbox_inches='tight')
    else:
        output_file = f'{output_dir}/Z{iZ}_trio_comparison.png'
        fig.savefig(output_file, dpi=150, bbox_inches='tight')
        print(f"Saved: {output_file}")
    
    plt.show()
    
    print(f"Z{iZ}: Data σ={np.nanstd(zval_data):.2f}, Model σ={np.nanstd(zval_model):.2f}, "
          f"Resid σ={np.nanstd(zval_residual):.2f} μm")


print("Helper functions loaded:")
print("  focal_plane_zernike_basis(), fit_focal_zernikes()")
print("  plot_fit_params_and_residuals(), plot_single_image_residual_grid(), plot_zernike_trio()")
print(f"\nField angle columns: thx_{coord_sys}, thy_{coord_sys}")
print(f"Zernike data column: zk_{coord_sys}")
print(f"Intrinsic column: zk_intrinsic_{coord_sys}")

In [ ]:
# Extract Zernike arrays and determine Noll indices from visit_info
zk_data = np.stack(aosTable[f'zk_{coord_sys}'])
zk_intrinsic_col = f'zk_intrinsic_{coord_sys}'
zk_model = np.stack(aosTable[zk_intrinsic_col])
nZk = zk_data.shape[1]

# Read Noll indices from visit_info (should be uniform across visits)
if visit_info is not None and 'nollIndices' in visit_info.colnames:
    # nollIndices is an array column — take the first row (all should be identical)
    noll_arr = np.array(visit_info['nollIndices'][0])
    iZs = [int(n) for n in noll_arr]
    if len(iZs) != nZk:
        print(f"WARNING: nollIndices length ({len(iZs)}) != zk array width ({nZk})")
        print(f"  nollIndices: {iZs}")
        print(f"  Falling back to contiguous range")
        iZs = list(range(4, 4 + nZk))
    else:
        print(f"Noll indices from visit_info: {iZs}")
else:
    # Fallback: assume contiguous starting at 4
    if nZk == 19:
        iZs = list(range(4, 23))
    else:
        iZs = list(range(4, 4 + nZk))
    print(f"Warning: nollIndices not in visit_info — assuming contiguous Z4-Z{iZs[-1]}")

iZidx = {iZ: i for i, iZ in enumerate(iZs)}

print(f"\nZernike arrays: shape {zk_data.shape} (rows x Zernikes)")
print(f"  Noll indices ({len(iZs)} terms): {iZs}")
print(f"  Coordinate system: {coord_sys}")
print(f"  Intrinsic column: {zk_intrinsic_col}")

# Determine which Zernikes to use for 3x4 subplot grids
iZs_plot_12 = iZs[:12] if len(iZs) >= 12 else iZs
iZs_plot_hi = iZs[12:]  # Higher-order: Z16,17,18,19,22,23,24,25,26
print(f"  Default 3x4 plot set: {iZs_plot_12}")
if iZs_plot_hi:
    print(f"  Higher-order plot set: {iZs_plot_hi}")

<a id='analysis'></a>
## Analysis

Filter for matched intra/extra donuts, then fit per-image linear model and create trio plots.

In [11]:
# Filter for matched intra/extra donuts
matched_mask = aosTable['matched_intra_extra']
print(f"Total donuts: {len(aosTable)}")
print(f"Matched intra/extra donuts: {np.sum(matched_mask)}")
print(f"Fraction matched: {np.sum(matched_mask)/len(aosTable):.3f}")

# Apply filter
aosTable_matched = aosTable[matched_mask]

Total donuts: 119711
Matched intra/extra donuts: 81038
Fraction matched: 0.677


<a id='fit'></a>
## Focal-Plane Zernike Fits per Image

Two fits per image and Zernike term:
- **z1toz3**: focal Noll Z1–Z3 (piston + tilt + tip) — linear model
- **z1toz6**: focal Noll Z1–Z6 (adds defocus + astigmatism) — quadratic model

Both use robust regression (Huber's T) and subtract the intrinsic model.

**Units note:** All fit coefficients have units of **μm** because the focal-plane Zernike
basis functions are dimensionless (field coordinates normalized to `fp_radius = 1.75°`).
The fit model is:

$$\text{wavefront}(x,y) = \sum_j c_j \, Z_j^{\text{focal}}(x/R,\; y/R)$$

so the actual wavefront variation at a given field point is $c_j$ times the basis function
value there. For example, focal Z5 $= 2\sqrt{6}\,xy/R^2$ peaks at $\approx 2\sqrt{6} \approx 4.9$,
so a coefficient $c_5 = 0.1\,\mu\text{m}$ corresponds to a wavefront change of
$0.1 \times 2\sqrt{6} \approx 0.49\,\mu\text{m}$ at the field point where Z5 is maximal.

In [ ]:
# Fit 1: focal Noll Z1-Z3 (piston + tilt + tip)
fit_table_z1toz3, zk_fit_z1toz3, zk_wgt_z1toz3 = fit_focal_zernikes(
    aosTable_matched, iZs, iZidx, coord_sys,
    max_focal_noll=3, include_intrinsic=True, prefix='z1toz3',
    zk_intrinsic_col=zk_intrinsic_col)

aosTable_matched['zk_fit_z1toz3'] = list(zk_fit_z1toz3)
aosTable_matched['zk_rlm_weights_z1toz3'] = list(zk_wgt_z1toz3)

# Also keep 'zk_fit' as alias for the primary (z1toz3) fit for backward compat
aosTable_matched['zk_fit'] = list(zk_fit_z1toz3)

print(f"\nSample z1toz3 (Z4): c1 range = [{fit_table_z1toz3['z1toz3_z4_c1'].min():.3f}, "
      f"{fit_table_z1toz3['z1toz3_z4_c1'].max():.3f}] μm")

# Fit 2: focal Noll Z1-Z6 (adds defocus + astigmatism)
fit_table_z1toz6, zk_fit_z1toz6, zk_wgt_z1toz6 = fit_focal_zernikes(
    aosTable_matched, iZs, iZidx, coord_sys,
    max_focal_noll=6, include_intrinsic=True, prefix='z1toz6',
    zk_intrinsic_col=zk_intrinsic_col)

aosTable_matched['zk_fit_z1toz6'] = list(zk_fit_z1toz6)
aosTable_matched['zk_rlm_weights_z1toz6'] = list(zk_wgt_z1toz6)

print(f"\nSample z1toz6 (Z4): c1 range = [{fit_table_z1toz6['z1toz6_z4_c1'].min():.3f}, "
      f"{fit_table_z1toz6['z1toz6_z4_c1'].max():.3f}] μm")
print(f"  c4 (defocus) range = [{fit_table_z1toz6['z1toz6_z4_c4'].min():.3f}, "
      f"{fit_table_z1toz6['z1toz6_z4_c4'].max():.3f}] μm")

# Flag bad fits: |coefficient| > threshold OR too few donuts
min_donuts_for_fit = 200
for pfx, ft in [('z1toz3', fit_table_z1toz3), ('z1toz6', fit_table_z1toz6)]:
    coeff_cols = [c for c in ft.colnames if c.startswith(f'{pfx}_z')
                  and '_c' in c and not c.endswith('_err') and not c.endswith('_scale')]
    coeff_arr = np.column_stack([np.array(ft[c]) for c in coeff_cols])
    bad_coeff = np.any(np.abs(coeff_arr) > bad_fit_threshold, axis=1)
    bad_ndonuts = np.array(ft['n_donuts']) < min_donuts_for_fit
    bad_mask = bad_coeff | bad_ndonuts
    ft[f'{pfx}_bad_fit'] = bad_mask
    n_bad = np.sum(bad_mask)
    n_bad_coeff = np.sum(bad_coeff & ~bad_ndonuts)
    n_bad_ndonuts = np.sum(bad_ndonuts)
    print(f"\n{pfx}: {n_bad}/{len(ft)} visits flagged as bad_fit")
    print(f"  {n_bad_coeff} with |coeff| > {bad_fit_threshold} μm, "
          f"{n_bad_ndonuts} with n_donuts < {min_donuts_for_fit}")
    if n_bad > 0:
        bad_rows = ft[bad_mask]
        for row in bad_rows:
            reasons = []
            if row['n_donuts'] < min_donuts_for_fit:
                reasons.append(f"n_donuts={row['n_donuts']}")
            for c in coeff_cols:
                if abs(row[c]) > bad_fit_threshold:
                    reasons.append(f"{c}={row[c]:.3f}")
            print(f"  day_obs={row['day_obs']} seq_num={row['seq_num']}  "
                  + ', '.join(reasons))

# Combined bad_fit flag for downstream use
fit_table = fit_table_z1toz3
fit_table['bad_fit'] = (fit_table_z1toz3['z1toz3_bad_fit']
                        | fit_table_z1toz6['z1toz6_bad_fit'])
n_bad_total = np.sum(fit_table['bad_fit'])
print(f"\nCombined: {n_bad_total}/{len(fit_table)} visits flagged as bad_fit")

In [ ]:
# Merge z1toz3 and z1toz6 fit tables, then merge with visit_info, and save

# Combine z1toz3 and z1toz6 fits (join on day_obs, seq_num)
z1toz6_cols_only = [c for c in fit_table_z1toz6.colnames if c.startswith('z1toz6_')]
fit_table_combined = fit_table_z1toz3.copy()
for col in z1toz6_cols_only:
    fit_table_combined[col] = fit_table_z1toz6[col]
# Add combined bad_fit flag
fit_table_combined['bad_fit'] = fit_table['bad_fit']

print(f"Combined fit table: {len(fit_table_combined)} rows, {len(fit_table_combined.columns)} columns")

# Merge with visit_info (if available)
if visit_info is not None:
    from astropy.table import join
    fit_merged = join(fit_table_combined, visit_info, keys=['day_obs', 'seq_num'],
                      join_type='left')
    print(f"Merged with visit_info: {len(fit_merged)} rows, {len(fit_merged.columns)} columns")
    print(f"  Visit_info columns added: {[c for c in visit_info.colnames if c not in ['day_obs', 'seq_num']]}")
else:
    fit_merged = fit_table_combined
    print("visit_info not available — skipping merge")

# Save as parquet
output_fits_file = (f'{run_dir}/intrinsic_fits_{wep_ver}_{dviz_ver}'
                    f'_{day_obs_min}_{day_obs_max}.parquet')
fit_merged.write(output_fits_file, format='parquet', overwrite=True)
print(f"\nSaved: {output_fits_file}")
print(f"  {len(fit_merged)} rows x {len(fit_merged.columns)} columns")
n_bad = np.sum(fit_merged['bad_fit'])
print(f"  {n_bad} visits flagged as bad_fit")

<a id='fitplots'></a>
## Fit Parameter Plots and Residual Histograms

Loop over each day_obs: show the first day inline, save all to output/.

In [ ]:
# Get list of day_obs values remaining after filtering
all_day_obs = sorted(set(np.array(aosTable_matched['day_obs']).tolist()))
print(f"Day_obs values for plots: {all_day_obs}")

# Build arrays needed for masking
day_obs_matched = np.array(aosTable_matched['day_obs'])
fit_day_obs = np.array(fit_table['day_obs'])

# Print pointing group summary (one-time, using good fits only)
if visit_info is not None:
    ft_good = fit_table[~fit_table['bad_fit']]
    _build_pointing_groups(ft_good, visit_info,
                           bin_size=position_group_tol, verbose=True)

In [ ]:
# Plot fit parameters and residual histograms for each day_obs
# First day shown inline; all days saved to run_dir as PDFs
# Make plots for both z1toz3 and z1toz6 fits
# Exclude bad-fit visits from plots

for iZs_set, set_label in [(iZs_plot_12, 'Z4-Z15'), (iZs_plot_hi, 'Z16+')]:
    if not iZs_set:
        continue
    print(f"\n{'='*60}")
    print(f"Fit parameter plots: {set_label} ({len(iZs_set)} Zernikes)")
    print(f"{'='*60}")
    
    for fit_prefix, ft in [('z1toz3', fit_table), ('z1toz6', fit_table_z1toz6)]:
        bad_col = f'{fit_prefix}_bad_fit'
        ft_good = ft[~ft[bad_col]] if bad_col in ft.colnames else ft
        ft_day_obs = np.array(ft_good['day_obs'])
        
        # Build mask excluding bad-fit visits from aosTable_matched
        bad_visits = set()
        if bad_col in ft.colnames:
            for row in ft[ft[bad_col]]:
                bad_visits.add((int(row['day_obs']), int(row['seq_num'])))
        if bad_visits:
            dobs_arr = np.array(aosTable_matched['day_obs'])
            snum_arr = np.array(aosTable_matched['seq_num'])
            good_donut_mask = np.array([
                (int(d), int(s)) not in bad_visits
                for d, s in zip(dobs_arr, snum_arr)
            ])
        else:
            good_donut_mask = np.ones(len(aosTable_matched), dtype=bool)
        
        for i, dobs in enumerate(all_day_obs):
            day_mask_fit = ft_day_obs == dobs
            day_mask_matched = (day_obs_matched == dobs) & good_donut_mask
            ft_day = ft_good[day_mask_fit]
            
            show_inline = (i == 0 and fit_prefix == 'z1toz3' and iZs_set is iZs_plot_12)
            plot_fit_params_and_residuals(
                ft_day, aosTable_matched, day_mask_matched,
                day_obs_list=[dobs], fit_prefix=fit_prefix,
                iZs_fit_plot=iZs_set, iZs_hist=iZs_set,
                visit_info=visit_info,
                zk_intrinsic_col=zk_intrinsic_col,
                output_dir=run_dir, show=show_inline)
        
        # Combined all-days plots
        show_inline = (fit_prefix == 'z1toz3' and iZs_set is iZs_plot_12)
        plot_fit_params_and_residuals(
            ft_good, aosTable_matched, good_donut_mask,
            day_obs_list=all_day_obs, fit_prefix=fit_prefix,
            iZs_fit_plot=iZs_set, iZs_hist=iZs_set,
            visit_info=visit_info,
            zk_intrinsic_col=zk_intrinsic_col,
            output_dir=run_dir, show=show_inline)

<a id='singleimage'></a>
## Single-Image Residual Maps

For each visit, plot a 4x3 grid of binned residual maps (Z4–Z15) using `binned_statistic_2d`.
Show the first visit inline; all visits are saved as JPEGs and combined into an MPEG movie.

In [ ]:
# Generate single-image residual maps for all visits
# First visit shown inline, rest saved to run_dir for movie

# Build lookups for band and pointing from visit_info (if available)
band_lookup = {}
pointing_lookup = {}
if visit_info is not None:
    for row in visit_info:
        key = (int(row['day_obs']), int(row['seq_num']))
        band_lookup[key] = str(row['band'])
        pointing_lookup[key] = {
            'alt': float(row['alt']),
            'az': float(row['az']),
            'rotAngle': float(row['rotAngle']),
        }

# Get sorted list of all (day_obs, seq_num) in matched table
all_images = sorted(set(zip(
    np.array(aosTable_matched['day_obs']).tolist(),
    np.array(aosTable_matched['seq_num']).tolist()
)))
print(f"Generating residual maps for {len(all_images)} visits...")
print(f"  Zernike subplot set: {iZs_plot_12}")

frame_files = []
for i, (dobs, snum) in enumerate(all_images):
    band = band_lookup.get((dobs, snum), '')
    ptg = pointing_lookup.get((dobs, snum), {})
    show_inline = (i == 0)
    outfile = plot_single_image_residual_grid(
        aosTable_matched, dobs, snum,
        band=band,
        alt=ptg.get('alt'), az=ptg.get('az'), rotAngle=ptg.get('rotAngle'),
        fit_table=fit_table, fit_prefix='z1toz3',
        iZs_plot=iZs_plot_12,
        zk_intrinsic_col=zk_intrinsic_col,
        n_steps=16, statistic='median',
        plo=2.0, phi=98.0,
        output_dir=run_dir, show=show_inline
    )
    if outfile is not None:
        frame_files.append(outfile)
    if (i + 1) % 50 == 0:
        print(f"  ...processed {i + 1}/{len(all_images)} visits")

print(f"Generated {len(frame_files)} residual map frames")

# Combine into MPEG movie using ffmpeg
movie_success = False
if len(frame_files) > 1:
    list_file = f'{run_dir}/frame_list.txt'
    with open(list_file, 'w') as f:
        for fpath in frame_files:
            f.write(f"file '{Path(fpath).name}'\n")
            f.write("duration 0.5\n")
    
    movie_file = f'{run_dir}/single_image_residuals.mp4'
    try:
        result = subprocess.run(
            ['ffmpeg', '-y', '-f', 'concat', '-safe', '0',
             '-i', 'frame_list.txt',
             '-vf', 'pad=ceil(iw/2)*2:ceil(ih/2)*2',
             '-c:v', 'libx264', '-pix_fmt', 'yuv420p',
             '-r', '2',
             'single_image_residuals.mp4'],
            capture_output=True, text=True, cwd=run_dir
        )
        if result.returncode == 0:
            print(f"Saved movie: {movie_file}")
            movie_success = True
        else:
            print(f"ffmpeg failed (return code {result.returncode}):")
            print(result.stderr[-500:] if len(result.stderr) > 500 else result.stderr)
    except FileNotFoundError:
        print("ffmpeg not found — movie not created. Install ffmpeg to generate MPEG.")

# Clean up individual JPEGs and frame_list.txt if movie was successfully created
if movie_success:
    for fpath in frame_files:
        Path(fpath).unlink(missing_ok=True)
    Path(list_file).unlink(missing_ok=True)
    print(f"Cleaned up {len(frame_files)} individual JPEGs and frame_list.txt")
else:
    print("Keeping individual JPEGs (movie was not created).")

<a id='viz'></a>
## Visualizations

Create trio plots for each Zernike term: Data (linear fit subtracted), Model, and Data-Model residuals.

In [ ]:
# Build day_obs label for trio plot titles
if len(all_day_obs) <= 4:
    day_obs_plot_str = 'day_obs: ' + ', '.join(str(d) for d in all_day_obs)
else:
    day_obs_plot_str = f'day_obs: {all_day_obs[0]}...{all_day_obs[-1]} ({len(all_day_obs)} days)'

# Exclude donuts belonging to bad-fit visits
bad_visits = set()
for row in fit_table:
    if row['bad_fit']:
        bad_visits.add((int(row['day_obs']), int(row['seq_num'])))

if bad_visits:
    dobs_arr = np.array(aosTable_matched['day_obs'])
    snum_arr = np.array(aosTable_matched['seq_num'])
    good_mask = np.array([
        (int(d), int(s)) not in bad_visits
        for d, s in zip(dobs_arr, snum_arr)
    ])
    aosTable_plot = aosTable_matched[good_mask]
    print(f"Excluded {len(bad_visits)} bad-fit visits, {np.sum(~good_mask)} donuts")
else:
    aosTable_plot = aosTable_matched

print(f"Donuts for trio plots: {len(aosTable_plot)} ({day_obs_plot_str})")

# All trio plots into a single PDF
trio_pdf_path = f'{run_dir}/trio_comparison_all.pdf'
with PdfPages(trio_pdf_path) as trio_pdf:
    for iZ in iZs:
        plot_zernike_trio(aosTable_plot, iZ=iZ, plo=plo_default, phi=phi_default,
                          statistic='median', fit_prefix='z1toz3',
                          zk_intrinsic_col=zk_intrinsic_col,
                          output_dir=run_dir, date_range_str=day_obs_plot_str,
                          pdf=trio_pdf)

print(f"\nSaved all trio plots to: {trio_pdf_path}")